# CRS data acquisition overview 
20260127: The CRS code is in the process of major revisions - this code may be out of date.

# Network setup 
Follow the steps in the rfmux Networking ReadME. Also, make sure "Use this connection only for resources on its network" is selected in the IPv4 settings for the ethernet port. 

In [ ]:
import os
import numpy as np
from citkid.crs.instrument import CRS
from citkid.crs.procedures import take_iq_noise
import warnings
warnings.filterwarnings("ignore", message="Discontinuous packet capture!")

crs = CRS(serial_number = 46, interface = 'enp2s0')
full_scale_dbm = 7
await crs.configure_system(clock_source="SMA", full_scale_dbm = full_scale_dbm,
                           verbose = True)

await crs.set_nco({1: 650e6, 2: 1149e6, 3: 1648e6, 4: 2147e6})
await crs.set_extended_module_bandwidth(extended = False)

In [ ]:
fres = 
# Optionally update fres by hand based on predicted shift from previous value
ares = 
qres = 
res_indices = 
fcal_indices = 
fres_all = 
qres_all = 

# Take IQ loops and noise
This first cell takes IQ loops, and optionally takes noise. To wait for user input after taking IQ loops and before taking noise, set "wait_for_noise" to True. 

In [ ]:
out_directory = '/home/<username>/data/<cooldown>/<dataset>/'
file_suffix = '00'

args = {'inst': crs,
        'fres': fres,
        'ares': ares,
        'qres': qres,
        'fcal_indices': fcal_indices,
        'res_indices': res_indices,
        'out_directory': out_directory,
        'file_suffix': file_suffix,
        'noise_time': 200,
        'take_noise': False,
        'gain_span_factor': 10,
        'npoints_noisefreq_update': 200,
        'npoints_fine': 500,
        'npoints_gain': 50,
        'npoints_rough': 100,
        'nsamps': 422, # If there is pulse tube noise, set this to the PT period
        'take_rough_sweep': False,
        'update_fres_from_fine': False,
        'fres_update_method': 'spacing',
        'fir_stage': 6,
        'fres_all': fres_all,
        'qres_all': qres_all,
        'cable_delay': 50e-9,
        'parser_loc': '/home/<username>/github/rfmux/firmware/r1.5.6/parser',
        'wait_for_noise': False # new parameter: pauses between IQ loops and  
                                # noise until user input is given
       }
args['npoints_fine'] = 10 
args['npoints_gain'] = 10

if os.path.exists(out_directory + f's21_fine_{file_suffix}.npy'):
    raise FileExistsError('File exists!')
await take_iq_noise(**args)

# Estimate file sizes for long timestreams

In [ ]:
from citkid.crs.util import estimate_timestream_data_size

fir_stage = 6
noise_time = 100 
nmodules = 4 
max_ntones = 1008 
# max_ntones returned from crs.write_tones, can be estimated by ntones / nmodules
ntones = 4000

estimate_timestream_data_size(fir_stage, noise_time, nmodules, max_ntones, ntones)

# If multiple noise measurements are desired, the following cell can be run.

## With batch processing

In [ ]:
# take noise data 
args = {'fres': fres,
        'ares': ares,
        'noise_time': 200,
        'fir_stage': 6,
        'fast_modules': [1],
        'parser_loc': '/home/<username>/github/rfmux/firmware/r1.5.6/parser',
        'batch_process': True,
        'outpath': out_directory + f'noise_{file_suffix}.npy',
        'batch_size': 500, # in MB. 
        'tmp_directory': 'tmp/'
    }
# batch_size is the amount loaded into memory at one time
# Note: files sizes will be batch_size * ntones / (max_ntones * nmodules)
await crs.capture_noise(**args) # saves the data directly

## Without batch processing

In [ ]:
# take noise data 
args = {'fres': fres,
        'ares': ares,
        'noise_time': 200,
        'fir_stage': 6,
        'fast_modules': [1],
        'parser_loc': '/home/<username>/github/rfmux/firmware/r1.5.6/parser',
        'batch_process': False
    }

z = await crs.capture_noise(**args)
# Without batch processing, capture_noise returns data instead of 
# saving it, so you need to save it here
np.save(out_directory + filename, [np.real(z), np.imag(z)])
fsample_noise = crs.sample_frequency
filename = f'noise_{file_suffix}_tsample_00.npy'
np.save(out_directory + filename, 1 / fsample_noise)

# Importing noise data

In [ ]:
# Without batch processing
i, q = np.load(out_directory + 'noise_00_00.npy')
z = i + 1j * q

In [ ]:
# With batch processing
from citkid.crs.util import import_noise_data
z = [] 
for batch_index in range(2): # loads first two files
    n_path = out_directory + f'noise_{file_suffix}_batch{batch_index:02d}.npy'
    sf_path = out_directory + f'noise_{file_suffix}_batch_scale_factor.npy'
    z.append(import_noise_data(n_path, sf_path))
z = np.concatenate(z, axis = 1)
dt = np.load(out_directory + f'noise_{file_suffix}_batch_tsample.npy')